# Phase 2 - Week 6 - Lesson Day 2
## Random Forests

Full explanation is in `Day2 Random Forests.pdf`. All answers are already solved, with comments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Made-up student data: will the student pass (1) or fail (0)?
np.random.seed(42)
n = 300
hours_studied = np.random.uniform(0, 10, n)
attendance_pct = np.random.uniform(50, 100, n)
sleep_hours = np.random.normal(7, 1.2, n)

# The "true" score depends on hours and attendance (sleep does not matter here)
score = hours_studied * 6 + attendance_pct * 0.3 + np.random.normal(0, 8, n)
passed = (score >= 55).astype(int)

df = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "sleep_hours": sleep_hours,
    "passed": passed,
})

X = df[["hours_studied", "attendance_pct", "sleep_hours"]]   # inputs
y = df["passed"]                                              # what we predict

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## Practice (Solved)

**1. SOLVED - Train a forest of 100 trees.**

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)   # builds 100 trees, each on a random piece of the data
print("Forest test score:", forest.score(X_test, y_test))

**2. SOLVED - Forest vs one tree.**

In [ ]:
one_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
one_tree.fit(X_train, y_train)

print("One tree:  ", one_tree.score(X_test, y_test))
print("Forest:    ", forest.score(X_test, y_test))
# On simple data like this, the two can be close. Always compare, never assume.

**3. SOLVED - Feature importance.** Bigger number = the forest used that column more.

In [ ]:
importance = pd.Series(forest.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))
# hours_studied is by far the biggest. attendance_pct and sleep_hours are small and close.
# In our made-up data, hours has a big effect, attendance a small one, sleep none.
# Sleep still gets a little importance because forests find small accidental patterns in noise,
# so treat these numbers as a rough guide, not exact truth.

**4. SOLVED - How many trees?**

In [ ]:
for trees in [5, 50, 300]:
    f = RandomForestClassifier(n_estimators=trees, random_state=42)
    f.fit(X_train, y_train)
    print(trees, "trees -> test score:", round(f.score(X_test, y_test), 3))
# The scores are almost the same. We only have 60 test students, so ONE student
# changes the score by about 0.017. Differences this small are mostly luck.
# More trees mainly makes the forest steadier from run to run.

**5. SOLVED - Predict a new student.**

In [ ]:
new_student = pd.DataFrame({
    "hours_studied": [6],
    "attendance_pct": [80],
    "sleep_hours": [7],
})

prediction = forest.predict(new_student)[0]
probability = forest.predict_proba(new_student)[0]

print("Prediction (1 = pass):", prediction)
print("Probability of fail, pass:", probability)